<a href="https://colab.research.google.com/github/kimdonggyu2008/Personal_Study/blob/main/las_%EA%B5%AC%ED%98%84_%EC%97%B0%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import abc
import yaml
import copy
import numpy as np
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence
import torch.nn.functional as F
import torchaudio

In [ ]:
BERT_FIRST_IDX = 997  # Replacing the 2 tokens right before english starts as <eos> & <unk>
BERT_LAST_IDX = 29635  # Drop rest of tokens

class _BaseTextEncoder(abc.ABC): #텍스트에 대한 인코딩
   def encode(self, s):
      raise NotImplementedError

    @abc.abstractmethod
    def decode(self, ids, ignore_repeat=False):
      raise NotImplementedError

    @abc.abstractproperty
    def vocab_size(self):
      raise NotImplementedError

    @abc.abstractproperty
    def token_type(self):
      raise NotImplementedError

    @abc.abstractclassmethod
    def load_from_file(cls, vocab_file):
      raise NotImplementedError

    @property
    def pad_idx(self):
      return 0

    @property
    def eos_idx(self):
      return 1

    @property
    def unk_idx(self):
      return 2

    def __repr__(self):
      return "<{} vocab_size={}>".format(type(self).__name__, self.vocab_size)




In [ ]:
class CharacterTextEncoder(_BaseTextEncoder):
    def __init__(self, vocab_list): #인코딩
        self._vocab_list = ["<pad>", "<eos>", "<unk>"] + vocab_list
        self.vocab2idx = {v: idx for idx, v in enumerate(self._vocab_list)}

    def encode(self, s):
        s = s.strip("\r\n")
        return [self.vocab_to_idx(c) for c in s] + [self.eos_idx]

    def decode(self, idxs, ignore_repeat=False):
        vocabs = []
        for t, idx in enumerate(idxs):
            v = self.idx_to_vocab(idx) #id 시퀀스에서 문자열로 변경
            if idx == self.pad_idx or (ignore_repeat and t > 0 and idx == idxs[t - 1]):
                continue
            elif idx == self.eos_idx:
                break
            else:
                vocabs.append(v)
        return "".join(vocabs)

    @classmethod
    def load_from_file(cls, vocab_file): #vocab 불러오기
        with open(vocab_file, "r", encoding="utf-8") as f:
            vocab_list = [line.strip("\r\n") for line in f]
        return cls(vocab_list)

    @property
    def vocab_size(self):
        return len(self._vocab_list)

    @property
    def token_type(self):
        return 'character'

    def vocab_to_idx(self, vocab): #문자 -> id
        return self.vocab2idx.get(vocab, self.unk_idx)

    def idx_to_vocab(self, idx):# id -> 문자
        return self._vocab_list[idx]


In [ ]:
class SubwordTextEncoder(_BaseTextEncoder): #서브 워드 단위 인코딩, 디코딩
    def __init__(self, spm):#단어보다 작은 단어 단위
        if spm.pad_id() != 0 or spm.eos_id() != 1 or spm.unk_id() != 2:
            raise ValueError(
                "Please train sentencepiece model with following argument:\n"
                "--pad_id=0 --eos_id=1 --unk_id=2 --bos_id=-1 --model_type=bpe --eos_piece=<eos>"
            )
        self.spm = spm

    def encode(self, s):
        return self.spm.encode_as_ids(s)

    def decode(self, idxs, ignore_repeat=False):
        crop_idx = []
        for t, idx in enumerate(idxs):
            if idx == self.eos_idx:
                break
            elif idx == self.pad_idx or (ignore_repeat and t > 0 and idx == idxs[t - 1]):
                continue
            else:
                crop_idx.append(idx)
        return self.spm.decode_ids(crop_idx)

    @classmethod
    def load_from_file(cls, filepath):
        import sentencepiece as splib
        spm = splib.SentencePieceProcessor()
        spm.load(filepath)
        spm.set_encode_extra_options(":eos")
        return cls(spm)

    @property
    def vocab_size(self):
        return len(self.spm)

    @property
    def token_type(self):
        return 'subword'


In [ ]:
class WordTextEncoder(CharacterTextEncoder):#단어 단위 인코더
  def encode(self,s):
    s=s.strip("\r\n")
    words=s.split(" ")
    return [self.vocab_to_idx(v) for v in words] + [self.eos_idx]

  def decode(self,idxs,ignore_repeat=False):
    vocabs=[]
    for t, idx in enumerate(idxs):
      v=self.idx_to_vocab(idx)
      if idx==self.eos_idx:
        break
      elif idx==self.pad_idx or (ignore_repeat and t>0 and idx==idxs[t-1]):
        continue
      else:
        vocabs.append(v)
    return " ".join(vocabs)

  def token_type(self):
    return 'word'

In [ ]:
class BertTextEncoder(_BaseTextEncoder):
  def __init__(self,tokenizer):
    self._tokenizer=tokenizer
    self._tokenizer.pad_token="<pad>"
    self._tokenizer.unk_token="<unk>"
    self._tokenizer.eos_token="<eos>"

  def encode(self,s):
    reduced_idx=[]
    for idx in self._tokenizer.encode(s):
      try:
        r_idx=idx-BERT_FIRST_IDX
        assert r_idx>0
        reduced_idx.append(r_idx)
      except:
        reduced_idx.append(self.unk_idx)
    reduced_idx.append(self.eos_idx)
    return reduced_idx

  def decode(self,idxs,ignore_repeat=False):
    crop_idx=[]
    for t,idx in enumerate(idxs):
      break
    elif idx==self.pad_idx or (ignore_repeat and t>0 and idx==idxs[t-1]):
      continue
    else:
      crop_idx.append(idx+BERT_FIRST_IDX)
    return self._tokenizer.decode(crop_idx)

  def vocab_size(self):
    return BERT_LAST_IDX-BERT_FIRST_IDX+1

  def token_type(self):
    return 'bert'

  def load_from_file(cls, vocab_file):
        from pytorch_transformers import BertTokenizer
        return cls(BertTokenizer.from_pretrained(vocab_file))

  @property
  def pad_idx(self):
      return 0

  @property
  def eos_idx(self):
      return 1

  @property
  def unk_idx(self):
      return 2

def load_text_encoder(mode, vocab_file):
    if mode == "character":
        return CharacterTextEncoder.load_from_file(vocab_file)
    elif mode == "subword":
        return SubwordTextEncoder.load_from_file(vocab_file)
    elif mode == "word":
        return WordTextEncoder.load_from_file(vocab_file)
    elif mode.startswith("bert-"):
        return BertTextEncoder.load_from_file(mode)
    else:
        raise NotImplementedError("`{}` is not yet supported.".format(mode))

In [ ]:
class CMVN(torch.jit.ScriptModule):
  def __init__(self,mode="global",dim=2,eps=1e-10):
    super(CMVN,self).__init__()
    if mode!="global":
      raise NotImplementedError(
                "Only support global mean variance normalization.")

    self.mode=mode
    self.dim=dim
    self.eps=eps

  def forward(self,x):
    if self.mode=="global":
      return (x-x.mean(self.dim,keepdim=True))/(self.eps+x.std(self.dim,keepdim=True))

  def extra_repr(self):
    return "mode={},dim={},eps={}".format(self.mode,self.dim,self.eps)


In [ ]:
class Delta(torch.jit.ScriptModule): #델타 특성
  __constants__=["order","window_size","padding"]

  def __init__(self,order=1,window_size=2):
    super(Delta,self).__init__()

    self.order=order
    self.window_size=window_size

    filters=self._create_filters(order,window_size)
    self.register_buffer("filters",filters)
    self.padding=(0,(filters.shape[-1]-1)//2)

  def forward(self,x):
    x=x.squeeze(0)
    return F.conv2d(x,weight=self.filters,padding=self.padding[0])

  def _create_filters(self,order,window_size):
    scales=[[1.0]]
    for i in range(1,order+1):
      prev_offset=(len(scales[i-1])-1)//2
      curr_offset=prev_offset+window_size

      curr=[0]*(len(scales[i-1])+2*window_size)
      normalizer=0.0
      for j in range(-window_size,window_size+1):
        normalizer+=j*j
        for k in range(-prev_offset,prev_offset+1):
          curr[j+k+curr_offset] += (j * scales[i-1][k+prev_offset])
      curr=[x/normalizer for x in curr]
      scales.append(curr)

    max_len=len(scales[-1])
    for i, scales in enumerate(scales[:-1]):
      padding=(max_len-len(scale))//2
      scales[i]=[0]*padding + scale+[0]*padding

    return torch.tensor(scales.unsqueeze(1).unsqueeze(1))

  def extra_repr(self):
    return "order={}, window_size={}".format(self.order, self.window_size)



In [ ]:
class Postproces(torch.jit.ScriptModule):
  def forward(self,x):
    x=x.permute(2,0,1)
    return x.reshape(x.size(0),-1).detach()

In [ ]:
# 공부하기

# TODO(Windqaq): make this scriptable
class ExtractAudioFeature(nn.Module):
    def __init__(self, mode="fbank", num_mel_bins=40, **kwargs):
        super(ExtractAudioFeature, self).__init__()
        self.mode = mode
        self.extract_fn = torchaudio.compliance.kaldi.fbank if mode == "fbank" else torchaudio.compliance.kaldi.mfcc
        self.num_mel_bins = num_mel_bins
        self.kwargs = kwargs

    def forward(self, filepath):
        waveform, sample_rate = torchaudio.load(filepath)

        y = self.extract_fn(waveform,
                            num_mel_bins=self.num_mel_bins,
                            channel=-1,
                            sample_frequency=sample_rate,
                            **self.kwargs)
        return y.transpose(0, 1).unsqueeze(0).detach()

    def extra_repr(self):
        return "mode={}, num_mel_bins={}".format(self.mode, self.num_mel_bins)


def create_transform(audio_config):
    feat_type = audio_config.pop("feat_type")
    feat_dim = audio_config.pop("feat_dim")

    delta_order = audio_config.pop("delta_order", 0)
    delta_window_size = audio_config.pop("delta_window_size", 2)
    apply_cmvn = audio_config.pop("apply_cmvn")

    transforms = [ExtractAudioFeature(feat_type, feat_dim, **audio_config)]

    if delta_order >= 1:
        transforms.append(Delta(delta_order, delta_window_size))

    if apply_cmvn:
        transforms.append(CMVN())

    transforms.append(Postprocess())

    return nn.Sequential(*transforms), feat_dim * (delta_order + 1)

In [ ]:
class VGCExtractor(nn.Module): #특징 추출기(이미지 기반)

  def __init__(self, input_dim):  # ← 오타 수정 필요 (nn.Module → input_dim)
    super(VGCExtractor, self).__init__()
    self.dim=64
    self.hide_dim=128
    in_channel, freq_dim, out_dim=self.check_dim(input_dim)
    self.in_channel=in_channel
    self.freq_dim=freq_dim
    self.out_dim=out_dim

    self.extractor=nn.Sequential( # b,1,128,80
        nn.Conv2d(in_channel,self.init_dim,3,stride=3,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.init_dim,self.init_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2) #절반으로 줄어듦?


        nn.Conv2d(self.init_dim,self.hide_dim,3,stride=1,padding=1),# kernel_size=3
        nn.ReLU(),
        nn.Conv2d(self.hide_dim,self.hide_dim,3,stride=1,padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,stride=2) # b,t//4,output_dim
    )

    def check_dim(self,input_dim):
      if input_dim%13==0: #MFCC
        return int(input_dim/13),13,(13//4)*self.hide_dim #TIME, 주파수 단위 크기, 특징 벡터 차원

      elif input_dim%40==0: #LOG MEL
        return int(input_dim/40),40,(40//4)*self.hide_dim

      else:
        raise ValueError('Invalid input dimension')

    def view_input(self,feature,feat_len): #b,t,d -> b,c,t,f
      feat_len=feat_len//4
      if feature.shape[1]%4!=0:
        feature=feature[:,:-(feature.shape[1]%4),:].contiguous()
      bs,ts,ds=feature.shape
      feature=feature.view(bs,ts,self.in_channel,self.freq_dim)
      feature=feature.transpose(1,2)
      return feature, feat_len


    def forward(self,feature, feat_len):
      feature, feat_len=self.view_input(feature,feat_len) #b,t,d -> b,c,t,f
      feature=feature.extractor(feature) #다운샘플링, 고차원 feqture 추출
      feature=feature.transpose(1,2) #b,c,t,f -> b,t,c,f
      feature=feature.contiguous().view(feature.shape[0],feature.shape[1],self.out_dim)
      return feature, feat_len

In [ ]:
class CNNExtractor(nn.Module):
  def __init__(self,input_dim,out_dim):
    super(CNNExtractor,self).__init__()
    self.out_dim=out_dim
    self.extractor=nn.Sequential(
        nn.Conv1d(input_dim,out_dim,4,stride=2,padding=1),
        nn.Conv1d(out_dim,out_dim,4,stride=2,padding=1),
    )

  def forward(self,feature,feat_len):
    feat_len=feat_len//4
    feature=feature.transpose(1,2)
    feature=self.extractor(feature)
    feature=feature.transpose(1,2)
    return feature, feat_len

In [ ]:
class RNNLayer(nn.Module):
  def __init__(self,input_dim,module,dim,bidirection,dropout,layer_norm,sample_rate,sample_style,proj):
    super(RNNLayer,self).__init__()
    rnn_out_dim=2*dim if bidirection else dim
    self.out_dim=sample_rate * rnn_out_dim if sample_rate>1 and sample_style=='concat' else rnn_out_dim
    self.dropout=dropout
    self.layer_norm=layer_norm
    self.sample_rate=sample_rate
    self.sample_style=sample_style
    self.proj=proj

    if self.sample_style not in ['drop','concat']:
      raise ValueError('Unsupported Sample Style: '+self.sample_style)

    self.layer=getattr(nn.module.upper())(
        input_dim,dim,bidirectional=bidirection,num_layer=1,batch_first=True
    )

    if self.layer_norm:
      self.ln=nn.LayerNorm(rnn_out_dim)
    if self.dropout>0:
      self.dp=nn.Dropout(p=dropout)
    if self.proj:
      self.pj=nn.Linear(rnn_out_dim,rnn_out_dim)

  def forward(self,input_x,x_len):
    if not self.training:
      self.layer.flatten_parameters()
    output,_=self.layer(input_x)

    if self.layer_norm:
      output=self.ln(output)

    if self.dropout>0:
      output=self.dp(output)

    if self.sample_rate>1:
      batch_size,timestep,feature_dim=output.shape
      x_len=x_len//self.sample_rate

      if self.sample_style='drop':
        output=output[:,::self.sample_rate,:].contiguous()
      else:
        if timestep%self.sample_rate!=0:
          output=output[:,:-(timestep%self.sample_rate),:]
        output=output.contiguous().view(batch_size,int(timestep/self.sample_rate),feature_dim*self.sample_rate)

    if self.proj:
      output=torch.tanh(self.pj(output))

    return output, x_len

In [ ]:
class BaseAttention(nn.Module):

  def __init__(self,temprature,num_head):
    super().__init__()
    self.temprature=temprature
    self.num_head=num_head
    self.softmax=nn.Softmax(dim=-1)
    self.reset_mem()

  def reset_mem(self):#마스크 초기화
    self.mask=None
    self.k_len=None

  def set_mem(self,prev_att):
    pass

  def compute_mask(self,k,k_len): #마스크 생성 함수
    self.k_len=k_len #b,t,d 패딩을 제외한 길이 확인
    bs,ts,_=k.shape # b,t
    self.mask=np.zeros((bs,self.num_head,ts)) #b, num_head, t
    for idx,sl in enumerate(k_len):# b,t,d
      self.mask[idx, : , sl:]=1 #마스크 범위 1로 변경
    self.maks=torch.from_numpy(self.mask).to(k_len.device,dtype=torch.bool).view(-1,ts)

  def _attend(self,energy, value): #마스크 씌우기
    attn=energy/self.temperature
    attn=attn.masked_fill(self.mask, -np.inf) #마스크 위치에 -inf 지정
    attn=self.softmax(attn)
    output=torch.bmm(attn.unsqueeze(1),value).squeeze(1)
    return output,attn

In [ ]:
class ScaleDotAttention(BaseAttention): # scaledot 연산

  def __init__(self,temperature,num_head):
    super().__init__(temperature,num_head)

  def forward(self,q,k,v):
    ts=k.shape[1]
    energy=torch.bmm(q.unsqueeze(1),k.transpose(1,2)).squeeze(1)#배치 단위 행렬곱
    output,attn=self._attend(energy,v)
    attn=attn.view(-1,self.num_head,ts)

    return output,attn

In [ ]:
class LocationAwareAttention(BaseAttention):
  def __init__(temperature,num_head):
    self.prev_att=None
    self.loc_conv=nn.Conv1d(num_head,kernel_num,kernel_size=2*kernel_size+1,padding=kernel_size, bias=False)
    self.gen_energy=nn.Linear(dim,1)
    self.dim=dim

  def reset_mem(self):
    super().reset_mem()
    self.prev_att=None

  def set_mem(self,prev_att):
    self.prev_att=prev_att

  def forward(self,q,k,v):
    bs_nj,ts,_=k.shape
    bs=bs_nh//self.num_head

    if self.prev_att is None:
      self.prev_att = torch.zeros((bs,self.num_head,ts)).to(k.device)
      for idx,sl in enumerate(self.k_len):
        self.prev_att[idx,:,:sl]=1.0/sl

    loc_content=torch.tanh(self.loc_proj(self.loc_conv(self.prev_att).transpose(1,2)))
    loc_content=loc_content.unsqueeze(1).repeat(1,self.num_head,1,1).view(-1,ts,self.dim)
    q=q.unsqueeze(1)

    energy=self.gen_energy(torch.tanh(k+q+loc_context)).squeeze(2)
    output,attn=self._attend(energy,v)
    attn=attn.view(bs,self.num_head,ts)
    self.prev_att=attn

    return output, attn

In [ ]:
class RNNLM(nn.Module):

  def __init__(self,vocab_size,emb_typing,emb_dim,module,dim,n_layers,dropout):
    super().__init__()
    self.dim=dim
    self.n_layers=n_layers
    self.emb_tying=emb_tying
    if emb_tying:
      assert emb_dim==dim
    self.vocab_size=vocab_size
    self.emb=nn.Embedding(vocab_size,emb_dim)
    self.dp1=nn.Dropout(dropout)
    self.dp2=nn.Dropout(dropout)
    self.rnn=getattr(nn,module.upper())(
        emb_dim,dim,num_layers=n_layers,dropout=dropout,batch_first=True
    )
    if not self.emb_tying:
      self.trans=nn.Linear(dim,vocab_size)

  def create_msg(self):
    msg = ['Model spec.| RNNLM weight tying = {}, # of layers = {}, dim = {}'.format(
            self.emb_tying, self.n_layers, self.dim)]
    return msg

  def forward(self,x,lens,hidden=None):
    emb_x=self.dp1(self.emb(x))
    if not self.training:
      self.rnn.flatten_parameters()
    packed=nn.utils.rnn_pack_padded_sequence(emb_x,lens,batch_first=True,enforce_sorted=False)
    outputs,hidden=self.rnn(packed,hidden)
    outputs,_=nn.utils.rnn.pad_packed_sequence(
        outputs,batch_first=True)
    if self.emb_tying:
      outputs=F.linear(self.dp2(outputs),self.emb.weight)
    else:
      outputs=self.trans(self.dp2(outputs))
    return outputs, hidden

In [ ]:
LOG_ZERO=-10000000.0

class CTPrefixScore():
  def __init__(self,x):
    self.logzero = -100000000.0
    self.blank = 0
    self.eos = 1
    self.x = x.cpu().numpy()[0]
    self.odim = x.shape[-1]
    self.input_length = len(self.x)

  def init_state(self):
    r=np.full((self.input_length,2),self.logzero,dtype=np.float32)

    r[0,1]=self.x[0,self.blank]
    for i in range(1,self.input_length):
      r[i,1]=r[i-1,1]+self.x[i,self.blank]
    return r

  def full_compute(self,g,r_prev):
    prefix_length=len(g)
    last_char=g[-1] if prefix_length>0 else 0

    r = np.full((self.input_length, 2, self.odim),
                    self.logzero, dtype=np.float32) #차원, 채울 값(log 0)

    start=max(1,prefix_length)

    if prefix_length==0:
      r[0,0,:] = self.x[0,:]

    psi = r[start-1, 0, :]

    phi = np.logaddexp(r_prev[:, 0], r_prev[:, 1])

    for t in range(start,self.input_length):
      prev_blank=np.full((self.odim),r_prev[t-1,1],dtype=np.float32) #시점 t에서 blank 출력한 경우
      prev_nonblank=np.full((self.odim),r_prev[t-1,0],dtype=np.float32) #시점 t에서 문자 출력한 경우
      prev_nonblank[last_char]=self.logzero

      phi = np.logaddexp(prev_nonblank, prev_blank)
      # P(h|current step is non-blank) = [ P(prev. step = y) + P()]*P(c)
      r[t, 0, :] = np.logaddexp(r[t-1, 0, :], phi) + self.x[t, :]
      # P(h|current step is blank) = [P(prev. step is blank) + P(prev. step is non-blank)]*P(now=blank)
      r[t, 1, :] = np.logaddexp(
          r[t-1, 1, :], r[t-1, 0, :]) + self.x[t, self.blank]
      psi = np.logaddexp(psi, phi+self.x[t, :])

    return psi,np.rollaxis(r,2)

  def cheap_compute(self, g, r_prev, candidates):
    '''Given prefix g, return the probability of all possible sequence y (where y = concat(g,c))
        This function considers only those tokens in candidates for c (memory efficient)'''
    prefix_length = len(g)
    odim = len(candidates)
    last_char = g[-1] if prefix_length > 0 else 0

    # init. r
    r = np.full((self.input_length, 2, len(candidates)),
                self.logzero, dtype=np.float32)

    # start from len(g) because is impossible for CTC to generate |y|>|X|
    start = max(1, prefix_length)

    if prefix_length == 0:
        r[0, 0, :] = self.x[0, candidates]    # if g = <sos>

    psi = r[start-1, 0, :]
    # Phi = (prev_nonblank,prev_blank)
    sum_prev = np.logaddexp(r_prev[:, 0], r_prev[:, 1])
    phi = np.repeat(sum_prev[..., None],odim,axis=-1)
    # Handle edge case : last tok of prefix in candidates
    if  prefix_length>0 and last_char in candidates:
        phi[:,candidates.index(last_char)] = r_prev[:,1]

    for t in range(start, self.input_length):
        # prev_blank
        # prev_blank = np.full((odim), r_prev[t-1, 1], dtype=np.float32)
        # prev_nonblank
        # prev_nonblank = np.full((odim), r_prev[t-1, 0], dtype=np.float32)
        # phi = np.logaddexp(prev_nonblank, prev_blank)
        # P(h|current step is non-blank) =  P(prev. step = y)*P(c)
        r[t, 0, :] = np.logaddexp( r[t-1, 0, :], phi[t-1]) + self.x[t, candidates]
        # P(h|current step is blank) = [P(prev. step is blank) + P(prev. step is non-blank)]*P(now=blank)
        r[t, 1, :] = np.logaddexp( r[t-1, 1, :], r[t-1, 0, :]) + self.x[t, self.blank]
        psi = np.logaddexp(psi, phi[t-1,]+self.x[t, candidates])

    # P(end of sentence) = P(g)
    if self.eos in candidates:
        psi[candidates.index(self.eos)] = sum_prev[-1]
    return psi, np.rollaxis(r, 2)



In [ ]:
class CTCHypothesis(): #ctc 가설 지정, 두 종류의 확률을 따로 관리함
  def __init__(self):
    self.y=[] #
    self.Pr_y_t_blank=0.0 # 마지막 토큰이 BLANK
    self.Pr_y_t_nblank=LOG_ZERO #마지막 토큰이 글자일 확률

    self.Pr_y_t_blank_bkup=0.0 # T-1 시점의 확률 보존용
    self.Pr_y_t_nblank_bkup=LOG_ZERO

    self.lm_output=None
    self.lm_hidden=None
    self.updated_lm=False

  def update_lm(self,outpu,hidden): #현재 가설에 대한 LM 모델 결과 저장
    self.lm_output=output
    self.lm_hidden=hidden
    self.updated_lm=True

  def get_len(self):
    return len(self.y)

  def get_string(self):
    return ''.join([str(s) for s in self.y])

  def get_score(self): #현재시점 T까지의 total log-prob 계산
    return np.logaddexp(self.Pr_y_t_blank,self.Pr_y_t_nblank)

  def get_final_score(self): #최종 스코어를 출력 길이로 정규화 한 점수
    if len(self.y)>0:
      return np.logaddexp(self.Pr_y_t_blank_bkup,self.Pr_y_t_nblank_bkup)
    else:
      return self.Pr_y_t_blank_bkup

  def check_same(self,y_2): #가설 y와 y2가 완전히 같은지 확인, 중복가설 제거
    if len(self.y)!=len(y_2):
      return False
    for i in range(len(self.y)):
      if self.y[i]!=y_2[i]:
        return False
    return True

  def update_Pr_nblank(self, ctc_y_t): #최종 마지막 토큰이 non-blank 일 확률 업데이트
        # ctc_y_t  : Pr(ye,t|x)
        # Pr+(y,t) = Pr+(y,t-1) * Pr(ye,t|x)
        self.Pr_y_t_nblank += ctc_y_t

  def update_Pr_nblank_prefix(self, ctc_y_t, Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix, Pr_ye_y=None):
    # ctc_y_t  : Pr(ye,t|x)
    lm_prob = Pr_ye_y if Pr_ye_y is not None else 0.0
    if len(self.y) == 0: return
    if len(self.y) == 1:
        Pr_ye_y_prefix = ctc_y_t + lm_prob + np.logaddexp(Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix)
    else:
        # Pr_ye_y : LM Pr(ye|y)
        Pr_ye_y_prefix = ctc_y_t + lm_prob + (Pr_y_t_blank_prefix if self.y[-1] == self.y[-2] \
                                    else np.logaddexp(Pr_y_t_blank_prefix, Pr_y_t_nblank_prefix))
    # Pr+(y,t) = Pr+(y,t) + Pr(ye,y^,t)
    self.Pr_y_t_nblank = np.logaddexp(self.Pr_y_t_nblank, Pr_ye_y_prefix)

  def update_Pr_blank(self, ctc_blank_t):
    # Pr-(y,t) = Pr(y,t-1) * Pr(-,t|x)
    self.Pr_y_t_blank = np.logaddexp(self.Pr_y_t_nblank_bkup, self.Pr_y_t_blank_bkup) + ctc_blank_t

  def add_token(self,token,):
    lm_prob=Pr_k_y if Pr_k_y is not None else 0.0
    if len(self.y) == 0:
      Pr_y_t_nblank_new = ctc_token_t + lm_prob + np.logaddexp(self.Pr_y_t_blank_bkup, self.Pr_y_t_nblank_bkup)
    else:
      # Pr_k_y : LM Pr(k|y)
      Pr_y_t_nblank_new = ctc_token_t + lm_prob + (self.Pr_y_t_blank_bkup if self.y[-1] == token else \
                                    np.logaddexp(self.Pr_y_t_blank_bkup, self.Pr_y_t_nblank_bkup))
    self.Pr_y_t_blank  = LOG_ZERO
    self.Pr_y_t_nblank = Pr_y_t_nblank_new

    self.Pr_y_t_blank_bkup  = self.Pr_y_t_blank
    self.Pr_y_t_nblank_bkup = self.Pr_y_t_nblank

    self.y.append(token)

  def orig_backup(self):
    self.Pr_y_t_blank_bkup  = self.Pr_y_t_blank
    self.Pr_y_t_nblank_bkup = self.Pr_y_t_nblank

In [ ]:
class CTCBeamDecoder(nn.Module):
  def __init__(self,asr,vocab_range,beam_size,vocab_candidate,lm_path='',lm_config='',lm_weight=0.0,device=None):
    super().__init__()
    self.asr=asr #ASR 모델
    self.vocab_range=vocab_range
    self.beam_size=beam_size
    self.vocab_candidate=vocab_candidate
    assert self.vocab_cand<=len(self.vocab_range)
    assert self.asr.enable_ctc

    self.apply_lm_weight>0
    self.lm_w=0
    if self.apply_lm: #디코더 값으로 학습 중간 확인
      self.device=device
      self.lm_w=lm_weight #LM의 가중치 비율
      self.lm_path=lm_path
      lm_config=yaml.load(open(lm_config,'r'),Loader=yaml.FullLoader)
      self.lm=RMMLM(self.asr.vocab_size,**lm_config['model']).to(self.device)#CTC 토큰 예측기
      self.lm.load_state_dict(torch.load(
          self.lm_path,map_location='cpu')['model'])

      self.lm.eval()

  def create_msg(self):
    msg = ['Decode spec| CTC decoding \t| Beam size = {} \t| LM weight = {}'.format(self.beam_size, self.lm_w)]
    return msg

  def forward(self,feat,feat_len):
    assert feat.shape[0] == 1, "Batchsize == 1 is required for beam search"

    ctc_output,encode_len,att_output,att_align,dec_state=self.asr(feat,feat_len,10)
    del encode_len, att_output,att_align,dec_state,feat_len
    ctc_output=F.log_softmax(ctc_output[0],dim=-1).cpu().detach().numpy()
    T=len(ctc_output) #CTC의 전체 길이

    B=[CTCHypothesis()]
    if self.apply_lm: #LM 적용 여부
    #LM 입력값 , TOKEN(B,T), LENGTH(T), HIDDEN_STATE(NONE)
    #출력값(B,T,V)
      output,hidden=self.lm(torch.zeros((1,1),dtype=torch.long).to(self.device),torch.ones(1,dtype=torch.long).to(self.device),None)
      B[0].update_lm((output).log_softmax(dim=-1).squeeze().cpu().numpy(),hidden)

    start=True
    for t in range(T):# 각 시간별
      if np.argmax(ctc_output[t])==0 and start: #PAD 토큰이 최빈값이면 해당 시점 무시
        continue
      else:
        start=False
      B_new=[]

      for i in range(len(B)): #배치 사이즈
        B_i_new=copy.deepcopy(B[i]) #새로운 가설을 만들기 위한 복사본
        if B_i_new.get_len()>0:
          if B_i_new.y[-1]==1:#현재 가설이 끝났다면(예측이 전부 다 됐다면)
            B_new.append(B_i_new) #끝났으면 후보에다가 추가
            continue
          B_i_new.update_Pr_nblank(ctc_output[t,B_i_new.y[-1]])#nblank 확률 업데이트

          for j in range(len(B)): #prefix 중복 가설 처리
            if i!=j and B[j].check_same(B_i_new.y[:-1]):
              lm_prob=0.0
              if self.apply_lm:
                lm_prob=self.lm_w*B[j].lm_output[B_i_new.y[-1]]
              B_i_new.update_Pr_nblank_prefix(ctc_output[t,B_i_new.y[-1]],#nblank 확률 업데이트
                                              B[j].Pr_y_t_blank,
                                              B[j].Pr_y_t_nblank,lm_prob)
              break

      B_i_new.update_Pr_blank(ctc_output[t,0]) #blank 확률 업데이트

      if self.apply_lm:
        lm_hidden=B_i_new.lm_hidden
        lm_probs=B_i_new.lm_output
      else:
        lm_hidden=None
        lm_probs=None

      if self.apply_lm: #lm이 적용되는 경우 후보를 정렬, soft fusion 방식
        ctc_vocab_cand=sorted(zip(
            self.vocab_range,ctc_output[t,self.vocab_ragne]+self.lm_w*lm_probs[self.vocab_range]),
                              reverse=True,key=lambda x:x[1])
      else: #다른 경우에 pure ctc 확률로 정렬
        ctc_vocab_cand=sorted(zip(self.vocab_range,ctc_output[t,self.vocab_range]),reverse=True,key=lambda x:x[1])

      for j in range(self.vocab_cand): # 각 후보별 내용 정리
        k=ctc_vocab_cand[j][0]
        hyp_yk=copy.deepcopy(B_i_new)
        lm_prob=0.0 if not self.apply_lm else self.lm_w*lm_probs[k]
        hyp_yk.add_token(k,ctc_output[t,kl],lm_prob)
        hyp_yk.updated_lm=False #lm 업데이트 여부
        B_new.append(hyp_yk)# 가설 추가
      B_i_new.orig_backup()
      B_i_new.append(B_i_new)
    del B
    B=[]

    B_new=sorted(B_new,key=lambda x:x.get_string()) #가설 정리
    B.append(B_new[0])
    for i in range(1,len(B_new)): #중복 가설 제거
      if B_new[i].check_same(B[-1].y): #동일한 경우
        if B_new[i].get_score()>B[-1].get_score():#더 가능성 높은 녀석을 추가
          B[-1]=B_new[i]
        continue
      else:
        B.append(B_new[i])
    del B_new

    if t==T-1:#마지막 시점에서는 정규화된 점수 사용
      B=sorted(B,reverse=True,key=lambda x:x.get_final_score())
    else:#중간 시점에서는 누적점수 사용
      B=sorted(B,reverse=True,key=lambda x:x.get_score())
    if len(B)>self.beam_size:#beam 갯수가 적으면 적은만큼만 사용
      B=B[:self.beam_size]

    if self.apply_lm and t<T-1:#lm 상태 업데이트
      for i in range(len(B)): #현재 가설의 마지막 토큰을 넣고 다음 토큰을 예측
        if B[i].get_len()>0 and not B[i].updated_lm:
          output,hidden=self.lm(B[i].y[-1]*torch.ones((1,1),dtype=torch.long).to(self.device),torch.ones(1,dtype=torch.long).to(self.device),B[i].lm_hidden)
          B[i].update_lm((output).log_softmax(dim=-1).squeeze().cpu().numpy(),hidden)

    return [b.y for b in B]